## Preparación de la data

### Librerías para Data Cleaning y carga de data

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

year_start = 2021

In [2]:
DF_races = pd.read_csv("f1_reduced/races.csv")
DF_circuits = pd.read_csv("f1_reduced/circuits.csv")

DF_drivers = pd.read_csv("f1_reduced/drivers.csv")
DF_constructors = pd.read_csv("f1_reduced/constructors.csv")

DF_status = pd.read_csv("f1_reduced/status.csv")
DF_pit_stops = pd.read_csv("f1_reduced/pit_stops.csv")
DF_sprint_results = pd.read_csv("f1_reduced/sprint_results.csv")
DF_results = pd.read_csv("f1_reduced/results.csv")

DF_constructor_results = pd.read_csv("f1_reduced/constructor_results.csv")

DF_qualifying = pd.read_csv("f1_reduced/qualifying.csv")

### Merging las tablas para un solo DataFrame

1. Uniendo las tablas `results` y `sprint_results`

In [3]:
# Diferenciar resultados de carreras normales y sprint
DF_results["in_sprint"] = 0
DF_sprint_results["in_sprint"] = 1

print(f"Tamaño 'results': {DF_results.shape[0]}  -  'sprint_results': {DF_sprint_results.shape[0]}")
all_results = pd.concat([DF_results, DF_sprint_results], ignore_index=False)

# añadiendo status
all_results = all_results.merge(DF_status, how="left", on="statusId", suffixes=("", "_status"))
all_results.drop(columns=["statusId"], inplace=True)
all_results.rename(columns={"rank": "fastestLap_position"}, inplace=True)

display(all_results.sample(10))
print(f"Tamaño sumado: {all_results.shape[0]}")

Tamaño 'results': 1799  -  'sprint_results': 360


,raceId,driverId,constructorId,number,grid,positionOrder,points,laps,time,milliseconds,fastestLap,fastestLap_position,fastestLapTime,fastestLapSpeed,in_sprint,status
433,1073,854,210,47,19,14,0.0,57,\N,\N,42,17.0,1:29.457,212.522,0,+1 Lap
464,1075,847,131,63,6,5,10.0,50,+32.732,5092025,43,7.0,1:32.821,239.454,0,Finished
1557,1132,847,131,63,1,19,0.0,33,\N,\N,3,19.0,1:31.298,232.289,0,Water pressure
184,1061,817,1,3,6,5,10.0,52,+42.624,7145908,51,8.0,1:31.284,232.325,0,Finished
1824,1065,844,6,16,8,6,0.0,18,+23.442,1697520,18,NaN,1:24.677,NaN,1,Finished
254,1064,9,51,88,16,15,0.0,70,\N,\N,44,14.0,1:15.442,203.234,0,+2 Laps
646,1084,846,1,4,10,7,6.0,70,\N,\N,62,8.0,1:09.304,224.298,0,+1 Lap
402,1072,822,131,77,2,3,15.0,50,+27.531,7602649,47,2.0,1:31.408,243.155,0,Finished
1664,1138,1,131,44,3,6,8.0,62,+1:25.248,6137819,43,12.0,1:37.393,182.600,0,Finished
1793,1144,862,214,61,17,15,0.0,57,\N,\N,56,13.0,1:29.121,213.323,0,+1 Lap


Tamaño sumado: 2159


2. Unir info de `drivers`, `constructors` y `constructor_results` 

In [4]:
# Constructor_results -> points por escudería
DF_constructor_results_reduced = DF_constructor_results.rename(columns={"points": "constructor_points"})
DF_constructor_results_reduced = DF_constructor_results_reduced[["raceId", "constructorId", "constructor_points"]]
all_results_p = all_results.rename(columns={"points": "pilot_points"})
added_cpoints_results = pd.merge(
    all_results_p,
    DF_constructor_results_reduced,
    how="left", on=["raceId", "constructorId"]
)
added_cpoints_results[["raceId", "constructorId", "pilot_points", "constructor_points"]].sample(5)       

,raceId,constructorId,pilot_points,constructor_points
1876,1077,3,0.0,0.0
150,1058,51,0.0,0.0
1550,1132,210,0.0,8.0
2083,1131,6,4.0,21.0
233,1063,1,0.0,6.0


In [5]:
def swap_columns(df, col1, col2):
    cols = df.columns.tolist()
    idx1, idx2 = cols.index(col1), cols.index(col2)
    cols[idx1], cols[idx2] = cols[idx2], cols[idx1]
    return df[cols]

In [6]:
# Drivers -> name y nationality  -  Constructors -> name
DF_drivers_reduced = DF_drivers[["driverId", "name"]].rename(columns={"name": "driver_name"})
DF_constructors_reduced = DF_constructors[["constructorId", "name"]].rename(columns={"name": "constructor_name"})
added_info_results = pd.merge(
    added_cpoints_results,
    DF_drivers_reduced,
    how="left", on=["driverId"]
)
added_info_results = pd.merge(
    added_info_results,
    DF_constructors_reduced,
    how="left", on=["constructorId"]
)
added_info_results = swap_columns(added_info_results, "driver_name", "driverId")
added_info_results = swap_columns(added_info_results, "constructor_name", "constructorId")
added_info_results.rename(columns={"positionOrder": "final_position"}, inplace=True)
added_info_results.sample(5)

,raceId,driver_name,constructor_name,number,grid,final_position,pilot_points,laps,time,milliseconds,fastestLap,fastestLap_position,fastestLapTime,fastestLapSpeed,in_sprint,status,constructor_points,driverId,constructorId
1948,1107,Lewis Hamilton,Mercedes,44,18,10,0.0,24,+46.375,1873105,24,NaN,1:10.775,NaN,1,Finished,11.0,1,131
735,1088,Guanyu Zhou,Alfa Romeo,24,14,16,0.0,72,+37.320,5840093,63,17.0,1:15.940,201.901,0,Finished,0.0,855,51
99,1056,Charles Leclerc,Ferrari,16,1,20,0.0,0,\N,\N,\N,0.0,\N,\N,0,Driveshaft,18.0,844,6
1600,1135,Max Verstappen,Red Bull,1,2,2,18.0,72,+22.896,5468415,30,9.0,1:14.752,205.110,0,Finished,26.0,830,9
1573,1133,Kevin Magnussen,Haas F1 Team,20,15,15,0.0,69,\N,\N,50,19.0,1:23.553,188.761,0,+1 Lap,0.0,825,210


3. Merging con `races`

In [7]:
# Primero unir races con circuitos -> circuitId
DF_circuits_reduced = DF_circuits[["circuitId", "name", "country"]].rename(columns={"name": "circuit_name", "country": "cir_country"})
races_with_circuits = pd.merge(
    DF_races,
    DF_circuits_reduced,
    how="left", on=["circuitId"]
)
races_with_circuits.drop(columns=["circuitId"], inplace=True)
races_with_circuits.sample(5)

,raceId,round,year,date,quali_date,sprint_date,circuit_name,cir_country
55,1110,12,2023,2023-07-30,2023-07-28,2023-07-29,Circuit de Spa-Francorchamps,Belgium
60,1115,17,2023,2023-10-08,2023-10-06,2023-10-07,Losail International Circuit,Qatar
33,1085,12,2022,2022-07-24,2022-07-23,\N,Circuit Paul Ricard,France
32,1084,11,2022,2022-07-10,2022-07-08,2022-07-09,Red Bull Ring,Austria
35,1087,14,2022,2022-08-28,2022-08-27,\N,Circuit de Spa-Francorchamps,Belgium


In [8]:
# Ahora si unir con los resultados -> raceId
added_races_results = pd.merge(
    added_info_results,
    races_with_circuits,
    how="left", on=["raceId"]
)
added_races_results.sample(5)

,raceId,driver_name,constructor_name,number,grid,final_position,pilot_points,laps,time,milliseconds,...,constructor_points,driverId,constructorId,round,year,date,quali_date,sprint_date,circuit_name,cir_country
1123,1111,Sergio Pérez,Red Bull,11,7,4,12.0,72,+10.068,8655093,...,37.0,815,9,13,2023,2023-08-27,2023-08-26,\N,Circuit Park Zandvoort,Netherlands
1709,1140,Lance Stroll,Aston Martin,18,14,11,0.0,70,\N,\N,...,0.0,840,117,20,2024,2024-10-27,2024-10-26,\N,Autódromo Hermanos Rodríguez,Mexico
1598,1134,George Russell,Mercedes,63,6,20,0.0,44,1:19:57.040,4797566,...,25.0,847,131,14,2024,2024-07-28,2024-07-27,\N,Circuit de Spa-Francorchamps,Belgium
456,1074,Nico Hülkenberg,Aston Martin,27,17,17,0.0,57,+1:03.829,5917413,...,0.0,807,117,1,2022,2022-03-20,2022-03-19,\N,Bahrain International Circuit,Bahrain
748,1089,Nyck de Vries,Williams,45,8,9,2.0,53,+7.122,4834633,...,2.0,856,3,16,2022,2022-09-11,2022-09-10,\N,Autodromo Nazionale di Monza,Italy


4. Merging con `qualifying`

In [9]:
DF_qualifying_reduced = DF_qualifying[["raceId", "driverId", "position", "q1", "q2", "q3"]] \
                        .rename(columns={"position": "qual_position"})
all_merged = pd.merge(
    added_races_results,
    DF_qualifying_reduced,
    how="left", on=["raceId", "driverId"]
)
# all_merged.drop(columns=["raceId"], inplace=True)
all_merged.sample(3)

,raceId,driver_name,constructor_name,number,grid,final_position,pilot_points,laps,time,milliseconds,...,year,date,quali_date,sprint_date,circuit_name,cir_country,qual_position,q1,q2,q3
111,1057,Valtteri Bottas,Mercedes,77,10,12,0.0,51,+11.264,8027674,...,2021,2021-06-06,2021-06-05,\N,Baku City Circuit,Azerbaijan,10.0,1:42.701,1:42.106,1:42.659
1675,1138,Pierre Gasly,Alpine F1 Team,10,18,17,0.0,61,\N,\N,...,2024,2024-09-22,2024-09-21,\N,Marina Bay Street Circuit,Singapore,18.0,1:31.312,\N,\N
890,1098,Yuki Tsunoda,AlphaTauri,22,14,11,0.0,57,+1:30.870,5727606,...,2023,2023-03-05,2023-03-04,\N,Bahrain International Circuit,Bahrain,14.0,1:31.400,1:32.510,\N


Tanto `driverId` como `constructorId` se mantienen como indices para visualizaciones dinámicas más adelante, al final reordenamos y renombramos algunas columnas quedando:

In [ ]:
all_merged.rename(columns={
    "time": "onRace_time",
    "date": "central_date",
    "q1": "q1_time",
    "q2": "q2_time",
    "q3": "q3_time",
    "grid": "grid_position",
    "round": "WeekEnd"
}, inplace=True)

col_order = [
    'driver_name',                                  # driver
    'constructor_name',                             # escudería
    "central_date", "quali_date", "sprint_date",    # fechas de carrera central, quali y sprint
    'in_sprint',                                    # sprint o no
    "q1_time", "q2_time", "q3_time",                # tiempos en los qualifying
    'qual_position',                                # posicion al final de qualifying (Q3 para formato normal Domingo, 
                                                    #                                  Sprint-Q3 para formato Sprint Sabado y 
                                                    #                                  Q3 dentro del formato Sprint para Domingo)
    'grid_position',                                # posición inicial en la carrera GP (no siempre igual a qualy)
    'onRace_time',                                  # tiempo en carrera
    'WeekEnd',                                      # N de Fin de semana respectivo
    'milliseconds',                                 # tiempo en ms
    'laps',                                         # nro de lap completadas
    'fastestLap',                                   # el lap con la vuelta más rápida 
    'fastestLapTime',                               # tiempo de la vuelta más rápida 
    'fastestLapSpeed',                              # velocidad de la vuelta más rápida                         -> (solo GP) <-
    'pilot_points',                                 # puntos al final del gp + sprint para el campeonato de pilotos
    'constructor_points',                           # puntos al final del gp + sprint para el campeonato de escuderías
    'fastestLap_position',                          # posición en la vuelta más rápida al final de la carrera   -> (solo GP) <-
    'final_position',                               # posición final de la carrera (individual del piloto)
    'status',                                       # status al final de la carrera 
    'circuit_name',                                 # circuito
    'cir_country',                                  # país del circuito
    'year',
    'constructorId',
    'driverId',
    "raceId"
]

f1_DF = all_merged[col_order].copy()
f1_DF.replace("\\N", np.nan, inplace=True)
# f1_DF.to_csv("F1_dataset.csv", index=False) # optional
f1_DF.sample(10)

,driver_name,constructor_name,central_date,quali_date,sprint_date,in_sprint,q1_time,q2_time,q3_time,qual_position,...,constructor_points,fastestLap_position,final_position,status,circuit_name,cir_country,year,constructorId,driverId,raceId
1948,Lewis Hamilton,Mercedes,2023-07-02,2023-06-30,2023-07-01,1,1:05.673,1:05.188,1:04.819,5.0,...,11.0,NaN,10,Finished,Red Bull Ring,Austria,2023,131,1,1107
1697,Guanyu Zhou,Sauber,2024-10-20,2024-10-19,2024-10-19,0,1:34.228,NaN,NaN,20.0,...,0.0,18.0,19,+1 Lap,Circuit of the Americas,USA,2024,15,855,1139
1934,Valtteri Bottas,Alfa Romeo,2023-04-30,2023-04-28,2023-04-29,1,1:42.582,1:42.259,NaN,14.0,...,0.0,NaN,16,Finished,Baku City Circuit,Azerbaijan,2023,51,822,1101
1621,Lando Norris,McLaren,2024-09-01,2024-08-31,NaN,0,1:19.911,1:19.727,1:19.327,1.0,...,34.0,1.0,3,Finished,Autodromo Nazionale di Monza,Italy,2024,1,846,1136
1352,Esteban Ocon,Alpine F1 Team,2024-03-09,2024-03-08,NaN,0,1:29.475,NaN,NaN,17.0,...,0.0,17.0,13,+1 Lap,Jeddah Corniche Circuit,Saudi Arabia,2024,214,839,1122
455,Nicholas Latifi,Williams,2022-03-20,2022-03-19,NaN,0,1:33.634,NaN,NaN,20.0,...,0.0,20.0,16,Finished,Bahrain International Circuit,Bahrain,2022,3,849,1074
1926,Lance Stroll,Aston Martin,2023-04-30,2023-04-28,2023-04-29,1,1:42.524,1:41.576,1:41.611,9.0,...,22.0,NaN,8,Finished,Baku City Circuit,Azerbaijan,2023,117,840,1101
479,Mick Schumacher,Haas F1 Team,2022-03-27,2022-03-26,NaN,0,1:30.167,1:29.920,NaN,14.0,...,2.0,0.0,20,Withdrew,Jeddah Corniche Circuit,Saudi Arabia,2022,210,854,1075
1661,Oscar Piastri,McLaren,2024-09-22,2024-09-21,NaN,0,1:30.258,1:29.640,1:29.953,5.0,...,40.0,4.0,3,Finished,Marina Bay Street Circuit,Singapore,2024,1,857,1138
280,Lewis Hamilton,Mercedes,2021-09-26,2021-09-25,NaN,0,1:45.992,1:45.129,1:44.050,4.0,...,35.0,2.0,1,Finished,Sochi Autodrom,Russia,2021,131,1,1066


## Data Cleaning

### Primero, familiarizando la F1

#### Campeonatos

En cada temporada se juegan dos torneos simultáneos. Esto explica por qué tienes ``pilot_points`` y ``constructor_points``.

- Campeonato de Pilotos (Driver's Championship): El piloto que suma más puntos al final del año es el Campeón Mundial.
- Campeonato de Constructores (Constructor's Championship): Se suman los puntos de los dos pilotos de la escudería.

#### Los WeekEnds

Primero tenemos el formato normal lo cual consite en:
1. Viernes: 
    
    - Free Practices FP1 y FP2, los equipos prueban el coche y la pista. No hay puntos, pero recogen datos para la estrategia.

2. Sábado: 

    - Una Free Practice FP3 
    - El Qualifying, se dividen en: Q1 (18 min), Q2 (15 min) y Q3 (12 min). Tras Q1 se eliminan los 5 más lentos, tras Q2 otros 5, y Q3 decide la pole entre los 10 restantes. La pole position determina la grid del Gran Premio del domingo (salvo penalizaciones).

3. Domingo:

    The Race: Aquí cuentan la resistencia, la estrategia y los neumáticos en el Grand Prix por puntos a los 10 primeros.

Y también tenemos en formato _Sprint_ lo cual consiste en:

1. Viernes:

    - Una Free Practice FP1
    - Sprint-Qualifying o Sprint Shootout de duraciones cortas: SQ1 (12 min), SQ2 (10 min) y SQ3 (8 min). Esta sesión establece la grid del Sprint.

2. Sábado:

    - Sprint (aprox. 100 km, entre 25–30 min) sin pit-stops obligatorios. Otorga puntos extra para el Gran Prix.
    - Qualifying para definir las grid del GP.

3. Domingo:

    The Race: De la misma manera que el formato normal a por los 10 primeros.

#### Consideraciones importantes

- En versiones antiguas (2021) el Sprint definía la grid de la carrera GP; en formatos actuales no siempre la condiciona.
- Los puntos del Grand Prix (clásico) están en orden de los 10 primeros puestos: 25–18–15–12–10–8–6–4–2–1. (Ojo, el punto por vuelta más rápida fue eliminado para 2025).
- En 2021 los puntos solo van a los 3 primeros del Sprint y ya desde 2022 en adelante, a los 8 primeros del Sprint.

### Cambios de formatos en las fechas y cronómetros.

In [11]:
def from_crono_to_secs(crono):
    # formato mm:ss.sss o ss.sss
    if pd.isna(crono):
        return np.nan
    parts = crono.split(":")
    if len(parts) == 2:
        minutes = int(parts[0])
        seconds = float(parts[1])
        total_seconds = minutes * 60 + seconds
        return round(total_seconds, 3)
    elif len(parts) == 1:
        return float(parts[0])
    else:
        return np.nan

In [12]:
trans_f1_DF = f1_DF.copy()

# los dates a formato datetime
trans_f1_DF["central_date"] = pd.to_datetime(trans_f1_DF["central_date"], format="%Y-%m-%d")
trans_f1_DF["quali_date"] = pd.to_datetime(trans_f1_DF["quali_date"], format="%Y-%m-%d")
trans_f1_DF["sprint_date"] = pd.to_datetime(trans_f1_DF["sprint_date"], format="%Y-%m-%d")

# de milliseconds a segundos
trans_f1_DF["q1_time"] = trans_f1_DF["q1_time"].apply(from_crono_to_secs).astype("Float32")
trans_f1_DF["q2_time"] = trans_f1_DF["q2_time"].apply(from_crono_to_secs).astype("Float32")
trans_f1_DF["q3_time"] = trans_f1_DF["q3_time"].apply(from_crono_to_secs).astype("Float32")
trans_f1_DF["fastestLapTime"] = trans_f1_DF["fastestLapTime"].apply(from_crono_to_secs).fillna(0.0).astype("float32")
trans_f1_DF["fastestLap_position"] = trans_f1_DF["fastestLap_position"].fillna(0).astype("int8")

Algunos son `str` que ¡deben ser numéricos!

In [13]:
trans_f1_DF["fastestLap"] = trans_f1_DF["fastestLap"].fillna(0).astype("int8")
trans_f1_DF["fastestLapSpeed"] = trans_f1_DF["fastestLapSpeed"].fillna(0.0).astype("float64")

Hay un asunto con `onRace_time` en base a cada fin de semana, vemos que el Pole aparece como tiempo establecido y los que siguen en tiempos de aumento respecto a este, cosa que en `milliseconds` aparecen todos fijos.

Mejor, tomar en cuenta el último, además usar `milliseconds` directamente la carga de bits es innecesaria; y de paso cambiar formato.

In [14]:
trans_f1_DF[["onRace_time", "milliseconds", "final_position"]].head(10)

,onRace_time,milliseconds,final_position
0,1:32:03.897,5523897,1
1,+0.745,5524642,2
2,+37.383,5561280,3
3,+46.466,5570363,4
4,+52.047,5575944,5
5,+59.090,5582987,6
6,+66.004,5589901,7
7,+67.100,5590997,8
8,+85.692,5609589,9
9,+86.713,5610610,10


In [15]:
trans_f1_DF.drop(columns=["onRace_time"], inplace=True)
trans_f1_DF["milliseconds"] = (
    trans_f1_DF["milliseconds"].fillna(0.0).astype("float32") / 60_000.0
).round(3)
trans_f1_DF.rename(columns={"milliseconds": "gp_time_mins"}, inplace=True)
trans_f1_DF.dtypes

driver_name                       str
constructor_name                  str
central_date           datetime64[us]
quali_date             datetime64[us]
sprint_date            datetime64[us]
in_sprint                       int64
q1_time                       Float32
q2_time                       Float32
q3_time                       Float32
qual_position                 float64
grid_position                   int64
WeekEnd                         int64
gp_time_mins                  float32
laps                            int64
fastestLap                       int8
fastestLapTime                float32
fastestLapSpeed               float64
pilot_points                  float64
constructor_points            float64
fastestLap_position              int8
final_position                  int64
status                            str
circuit_name                      str
cir_country                       str
year                            int64
constructorId                   int64
driverId    

¡Espera!, hay un problema en la carrera GP de `raceId=1134`, los tiempos estan muy sin sentido, ordenaremos correctamente.

In [16]:
idx_race1134 = trans_f1_DF["raceId"] == 1134
race1134 = trans_f1_DF[idx_race1134][["driver_name", "gp_time_mins"]]
race1134

,driver_name,gp_time_mins
1579,Lewis Hamilton,0.009
1580,Oscar Piastri,0.028
1581,Charles Leclerc,0.151
1582,Max Verstappen,0.163
1583,Lando Norris,0.173
1584,Carlos Sainz,0.339
1585,Sergio Pérez,0.729
1586,Fernando Alonso,0.841
1587,Esteban Ocon,0.885
1588,Daniel Ricciardo,0.924


In [17]:
race1134_races_times = race1134["gp_time_mins"].to_list()
race1134_best_time = race1134_races_times[-1]
race1134_races_times.pop()
race1134_races_times.insert(0, 0)
race1134_races_times = [i + race1134_best_time for i in race1134_races_times]

trans_f1_DF.loc[idx_race1134, "gp_time_mins"] = race1134_races_times
trans_f1_DF[idx_race1134][["driver_name", "gp_time_mins"]]

,driver_name,gp_time_mins
1579,Lewis Hamilton,79.959000
1580,Oscar Piastri,79.968002
1581,Charles Leclerc,79.987000
1582,Max Verstappen,80.110001
1583,Lando Norris,80.122002
1584,Carlos Sainz,80.131996
1585,Sergio Pérez,80.297997
1586,Fernando Alonso,80.687996
1587,Esteban Ocon,80.800003
1588,Daniel Ricciardo,80.844002


### Veamos los datos faltantes, ¿a qué se deben?

- Primero, los NA's en las columnas `fastestLap_position` y `fastestLapSpeed`, que ahora son ceros, son debido a que provienen de las carreras Sprint y no existen esas variables.

In [18]:
# separando por in_sprint
f1_DF_sprint = trans_f1_DF[trans_f1_DF["in_sprint"] == 1].copy()
f1_DF_gp = trans_f1_DF[trans_f1_DF["in_sprint"] == 0].copy()

# Ahora analizar los datos faltantes
print(f"Datos faltantes en formato Sprint (size = {f1_DF_sprint.shape[0]}):")
for col_name in f1_DF_sprint.columns:
    missing_count = f1_DF_sprint[col_name].isna().sum()
    if missing_count > 0:
        print(f"  {col_name:<20}: {missing_count}")

print(f"\nDatos faltantes en formato GP (size = {f1_DF_gp.shape[0]}):")
for col_name in f1_DF_gp.columns:
    missing_count = f1_DF_gp[col_name].isna().sum()
    if missing_count > 0:
        print(f"  {col_name:<20}: {missing_count}")

# los sizes divididos sale igual a los sizes de la carga inicial de results + sprint results

Datos faltantes en formato Sprint (size = 360):
  q1_time             : 2
  q2_time             : 91
  q3_time             : 187

Datos faltantes en formato GP (size = 1799):
  sprint_date         : 1439
  q1_time             : 21
  q2_time             : 465
  q3_time             : 921
  qual_position       : 1


- Claramente los `sprint_dates` faltantes vienen del GP porque son normales, solo están los GP en domingos de las carreras en formato Sprint.


In [19]:
missing_qual = trans_f1_DF.apply(
    lambda row: pd.isna(row["q1_time"]) or pd.isna(row["q2_time"]) or pd.isna(row["q3_time"]), 
    axis=1
)
missing_qual_rows = trans_f1_DF[missing_qual]
# _temp_cols = ["driver_name", "constructor_name", "q1_time", "q2_time", "q3_time", "qual_position"]
print("Las posiciones con qtimes faltantes:")
print(missing_qual_rows["qual_position"].unique().tolist())

Las posiciones con qtimes faltantes:
[11.0, 13.0, 14.0, 12.0, 16.0, 15.0, 18.0, 19.0, 17.0, 20.0, 10.0, nan, 9.0]


- Basicamente a partir del 10mo puesto de las Q's eliminatorios, la data no presenta los tiempos de los excluidos (y descalificados por los `q1_time`'s nulos). Al final, como Qualifyings solo es una definición para las posiciones grid para el Grand Prix, solo se analiza estas variables en plano general en el tiempo de carreras.

### Vamos con Feature Engineering !!

**Frecuencia de pilotos por temporada**

Nota: puede que la frecuencia sea mayor de los dias de carrera, ya que se están contando los días de _Sprint_.

In [20]:
max_year = DF_races["year"].max()

trans_f1_DF["seasonFrec"] = ""
count_races_year = DF_races.groupby("year").size()
for year in range(year_start, max_year + 1):
    idx_f1_df_year = trans_f1_DF["year"] == year
    f1_df_year = trans_f1_DF[idx_f1_df_year]
    driver_counts_year = f1_df_year['driverId'].value_counts()
    f1_df_year['seasonFrec'] = f1_df_year['driverId'].apply(
        lambda dr: 'FullTime' if driver_counts_year[dr] >= count_races_year[year] else 'PartTime'
    )
    trans_f1_DF.loc[idx_f1_df_year, 'seasonFrec'] = f1_df_year['seasonFrec'] + " " + str(year)

trans_f1_DF[["driver_name", "seasonFrec"]]

,driver_name,seasonFrec
0,Lewis Hamilton,FullTime 2021
1,Max Verstappen,FullTime 2021
2,Valtteri Bottas,FullTime 2021
3,Lando Norris,FullTime 2021
4,Sergio Pérez,FullTime 2021
...,...,...
2154,Liam Lawson,PartTime 2024
2155,Yuki Tsunoda,FullTime 2024
2156,Franco Colapinto,PartTime 2024
2157,Guanyu Zhou,FullTime 2024


**Separación de fecha y hora**

In [21]:
trans_f1_DF["mth_gp"] = trans_f1_DF["central_date"].dt.month
trans_f1_DF["day_gp"] = trans_f1_DF["central_date"].dt.day
trans_f1_DF["mth_sprint"] = trans_f1_DF["sprint_date"].dt.month
trans_f1_DF["day_sprint"] = trans_f1_DF["sprint_date"].dt.day
trans_f1_DF["mth_quali"] = trans_f1_DF["quali_date"].dt.month
trans_f1_DF["day_quali"] = trans_f1_DF["quali_date"].dt.day

trans_f1_DF.drop(columns=["central_date", "sprint_date", "quali_date"], inplace=True)

In [22]:
trans_f1_DF.columns

Index(['driver_name', 'constructor_name', 'in_sprint', 'q1_time', 'q2_time',
       'q3_time', 'qual_position', 'grid_position', 'WeekEnd', 'gp_time_mins',
       'laps', 'fastestLap', 'fastestLapTime', 'fastestLapSpeed',
       'pilot_points', 'constructor_points', 'fastestLap_position',
       'final_position', 'status', 'circuit_name', 'cir_country', 'year',
       'constructorId', 'driverId', 'raceId', 'seasonFrec', 'mth_gp', 'day_gp',
       'mth_sprint', 'day_sprint', 'mth_quali', 'day_quali'],
      dtype='str')

Finalizando con cierta cosas en el DataFrame y guardarlo.

In [26]:
# De paso reordenar columnas
new_col_order = ['driver_name', 'constructor_name', 'year', 'pilot_points', 'constructor_points', 
                 'final_position', 'status', 'WeekEnd', 'circuit_name', 'cir_country', 'gp_time_mins',
                 'q1_time', 'q2_time', 'q3_time', 'qual_position', 'grid_position',
                 'laps', 'fastestLap', 'fastestLapTime', 'fastestLapSpeed', 'fastestLap_position',
                 'mth_gp', 'day_gp', 'mth_sprint', 'day_sprint', 'mth_quali', 'day_quali', 'in_sprint',
                 'seasonFrec','constructorId', 'driverId', 'raceId']

final_DF = trans_f1_DF[new_col_order].copy()
final_DF = final_DF.sort_values(
    ["year", "mth_gp", "raceId", "pilot_points"],
    ascending=[True, True, True, False]
).reset_index(drop=True)
final_DF.to_csv("F1_cleaned.csv", index=False)
final_DF.head(10)

,driver_name,constructor_name,year,pilot_points,constructor_points,final_position,status,WeekEnd,circuit_name,cir_country,...,day_gp,mth_sprint,day_sprint,mth_quali,day_quali,in_sprint,seasonFrec,constructorId,driverId,raceId
0,Lewis Hamilton,Mercedes,2021,25.0,41.0,1,Finished,1,Bahrain International Circuit,Bahrain,...,28,NaN,NaN,3,27,0,FullTime 2021,131,1,1052
1,Max Verstappen,Red Bull,2021,18.0,28.0,2,Finished,1,Bahrain International Circuit,Bahrain,...,28,NaN,NaN,3,27,0,FullTime 2021,9,830,1052
2,Valtteri Bottas,Mercedes,2021,16.0,41.0,3,Finished,1,Bahrain International Circuit,Bahrain,...,28,NaN,NaN,3,27,0,FullTime 2021,131,822,1052
3,Lando Norris,McLaren,2021,12.0,18.0,4,Finished,1,Bahrain International Circuit,Bahrain,...,28,NaN,NaN,3,27,0,FullTime 2021,1,846,1052
4,Sergio Pérez,Red Bull,2021,10.0,28.0,5,Finished,1,Bahrain International Circuit,Bahrain,...,28,NaN,NaN,3,27,0,FullTime 2021,9,815,1052
5,Charles Leclerc,Ferrari,2021,8.0,12.0,6,Finished,1,Bahrain International Circuit,Bahrain,...,28,NaN,NaN,3,27,0,FullTime 2021,6,844,1052
6,Daniel Ricciardo,McLaren,2021,6.0,18.0,7,Finished,1,Bahrain International Circuit,Bahrain,...,28,NaN,NaN,3,27,0,FullTime 2021,1,817,1052
7,Carlos Sainz,Ferrari,2021,4.0,12.0,8,Finished,1,Bahrain International Circuit,Bahrain,...,28,NaN,NaN,3,27,0,FullTime 2021,6,832,1052
8,Yuki Tsunoda,AlphaTauri,2021,2.0,2.0,9,Finished,1,Bahrain International Circuit,Bahrain,...,28,NaN,NaN,3,27,0,FullTime 2021,213,852,1052
9,Lance Stroll,Aston Martin,2021,1.0,1.0,10,Finished,1,Bahrain International Circuit,Bahrain,...,28,NaN,NaN,3,27,0,FullTime 2021,117,840,1052


### Adicional: Laps and pit-stops

Los datos solo presentan los tiempos Laps y pit-stops de los pilotos en las carreras centrales sin _Sprint_ en ciertos `raceId`.

In [27]:
lap_times = pd.read_csv("f1_reduced/lap_times.csv")
pit_stops = pd.read_csv("f1_reduced/pit_stops.csv")

lap_times["milliseconds"] = (
    lap_times["milliseconds"].fillna(0.0).astype("float32") / 60_000.0  # 60 sec * 1000 mill
).round(3)
lap_times.rename(columns={'milliseconds': "duration_lap_mins", "position": "position_on_lap"}, inplace=True)

pit_stops["milliseconds"] = (
    pit_stops["milliseconds"].fillna(0.0).astype("float32") / 1000.0   # 1000 mill
).round(3)
pit_stops.rename(columns={'milliseconds': "duration_stop_secs"}, inplace=True)

In [28]:
result = pd.merge(lap_times, pit_stops, on=["raceId", "driverId", "lap"], how="left")
result["duration_stop_secs"] = result["duration_stop_secs"].fillna(0.0)
result.sort_values(by=["raceId", "driverId", "lap"])
result.drop(columns=["lap"], inplace=True)
result.to_csv("lap_stops_gp.csv", index=False)
result.sample(5)

,raceId,driverId,position_on_lap,duration_lap_mins,duration_stop_secs
67959,1117,822,13,1.403,0.0
60452,1110,847,6,1.900,0.0
30395,1080,846,6,1.245,0.0
58365,1108,840,11,1.528,0.0
37912,1087,832,3,1.866,0.0
